# Notebook 8: The `sbikit` Package

In Notebook 7 we wrote a complete analysis by hand: a compression network, a training loop for the likelihood, a batched ensemble sampler and a coverage test. [`sbikit`](https://github.com/D-Gebauer/sbikit) is a package which provides exactly this pipeline, designed for cosmological analyses with neural likelihood estimation:

1. **Compression** of the data vector to a few summaries (VMIM, MOPED, and others),
2. **Neural likelihood estimation** of the summaries with a normalizing flow,
3. **Sampling** of the posteriors of hundreds of observations in one batched ensemble MCMC, with convergence checks,
4. **Metrics**: TARP, marginal coverage and figures of merit, with bootstrap errors.

Each piece works on its own with NumPy arrays in and out. In this notebook we repeat the analysis of Notebook 7 on the same simulations. It is both a continuation of the tutorial and a quickstart for the package.

**What you will learn**
- the `sbikit` workflow: compressor, `NLE`, `Prior`, `sample_posteriors`, `evaluate`
- how to read the convergence diagnostics of the sampler
- how to quantify calibration with numbers, not only with a plot, and what a more sensitive test reveals
- how to change the prior after sampling, and how to save the results

**Runtime:** about three minutes on a laptop.

*Version note:* this notebook was written for `sbikit` 0.1.0, which requires `sbi` 0.27 and `zuko` 1.6. The documentation is the README of the repository, and every public class and function has a docstring.

In [ ]:
import time
import tempfile
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import sbikit as sk
from sbikit.compress import VMIMCompressor
from sbikit.metrics import evaluate, importance_weights, posterior_summary

import emcee
from chainconsumer import ChainConsumer, Chain, Truth, PlotConfig

# The toy problem from Notebook 7
from tutorial_utils import lognormal_benchmark, exact_log_posterior, LOGNORMAL_PRIOR_LOW, LOGNORMAL_PRIOR_HIGH

np.random.seed(42)
torch.manual_seed(42)

# Our models are small, so they do not benefit from many CPU threads. Limiting the thread
# count avoids large slow-downs from thread oversubscription (common on laptops).
torch.set_num_threads(4)

## 1. The Simulations

We use the log-normal toy model and the fixed set of simulations from Notebook 7: three parameters, a data vector with 30 entries, and training simulations which do **not** follow the prior (half of them are concentrated in the inner cube $[-1, 1]^3$).

`sbikit` does not load data or run simulators. It takes arrays (NumPy arrays or PyTorch tensors) and returns NumPy arrays. The convention is the same everywhere: data vectors `x` have the shape `[N, D]`, parameters `theta` the shape `[N, P]`.

In [ ]:
benchmark = lognormal_benchmark()
param_names = ['theta1', 'theta2', 'theta3']

# Training and validation data for the compression (do not follow the prior)
x_train_npe, theta_train_npe = benchmark.x_train_npe, benchmark.theta_train_npe
x_val_npe, theta_val_npe = benchmark.x_val_npe, benchmark.theta_val_npe
# Training and validation data for the likelihood (do not follow the prior)
x_train_nle, theta_train_nle = benchmark.x_train_nle, benchmark.theta_train_nle
x_val_nle, theta_val_nle = benchmark.x_val_nle, benchmark.theta_val_nle
# Test data (follow the prior) and the observation
num_test = 300
x_test, theta_test = benchmark.x_test[:num_test], benchmark.theta_test[:num_test].numpy()
x_obs, theta_obs = benchmark.x_obs, benchmark.theta_obs

print("Training data for the compression:", tuple(x_train_npe.shape), tuple(theta_train_npe.shape))
print("Training data for NLE:            ", tuple(x_train_nle.shape), tuple(theta_train_nle.shape))
print("Test data:                        ", tuple(x_test.shape), theta_test.shape)

Our simulations are already split into training, validation and test sets. If yours are not, `sbikit` has two small helpers: `Dataset` holds the simulations together with the parameter names, and `resolve_splits` draws random splits (or checks the ones you provide, e.g. that they do not overlap). How the simulations are split remains your decision.

In [ ]:
dataset = sk.Dataset(theta_train_nle, x_train_nle, param_names=param_names)     # note the order: parameters first
splits = sk.resolve_splits(dataset, fractions={'train': 0.8, 'val': 0.1, 'test': 0.1}, seed=0)
print({name: len(splits[name]) for name in ('train', 'val', 'test')})

## 2. Compression

All compressors in `sbikit.compress` share one interface: `fit(...)` trains them, and calling the object compresses data vectors `[N, D]` to summaries `[N, n_out]`.

| Compressor | Method |
|---|---|
| `VMIMCompressor` | network trained with the VMIM loss (Notebook 7) |
| `MSECompressor` | network trained to predict the parameters (mean squared error) |
| `MOPED` | optimal *linear* compression from simulations at a fiducial point and their derivatives |
| `IMNN` | information-maximising neural network |
| `HierarchicalVMIM` | VMIM for data vectors which consist of several blocks |

We use VMIM, as in Notebook 7. The network and its training are configured with a dictionary (see `sk.NetConfig` for all options and their defaults). The defaults are meant for larger problems, so we choose a smaller network and fewer epochs here.

(MOPED would be a poor choice for this problem: it is linear and uses the derivatives of the mean data vector at the fiducial point, and the derivative with respect to $\theta_2$ vanishes at $\theta = 0$ because the data depend on $\theta_2^2$.)

In [ ]:
start = time.time()
compressor = VMIMCompressor(
    n_out=6,                                  # number of summaries
    config={
        'widths': (64, 64),                   # hidden layers of the compression network
        'epochs': 150,
        'lr': 2e-3,
        'batch_size': 128,
    },
    device='cpu',
    seed=0,
)
# Arguments: data and parameters of the training set, then of the validation set. Returns the best validation loss.
val_loss = compressor.fit(x_train_npe, theta_train_npe, x_val_npe, theta_val_npe)
print(f'VMIM: best validation loss {val_loss:.3f} after {len(compressor.history)} epochs ({time.time() - start:.0f} s)')

# Compress all data sets: calling the compressor returns NumPy arrays [N, 6]
s_train = compressor(x_train_nle)
s_val = compressor(x_val_nle)
s_test = compressor(x_test)
s_obs = compressor(x_obs[None])               # the observation, with a batch dimension: [1, 6]
print("Shape of the compressed training data:", s_train.shape)

plt.plot(compressor.history)
plt.xlabel('Epoch')
plt.ylabel('Validation loss')
plt.title('VMIM compression')
plt.show()

The compressor takes care of the normalization of the data internally, so there is no trap like the one in Notebook 7, where we had to remember to apply the standardization before the embedding network.

## 3. Neural Likelihood Estimation

`sk.NLE` is a conditional flow for the likelihood of the summaries, $q(s|\theta)$. The flow is configured with the dictionary `flow` (see `sk.FlowConfig`), the training with the dictionary `train` in `fit` (see `sk.TrainConfig`): Adam, a learning rate which is reduced when the validation loss stops improving, and early stopping, like our hand-written loop in Notebook 7. We use the same flow as there.

In [ ]:
start = time.time()
torch.manual_seed(0)       # the weights of the flow are initialized when the NLE object is created
nle = sk.NLE(
    n_x=6,                 # dimension of the summaries
    n_theta=3,             # number of parameters
    flow={'model': 'nsf', 'hidden_features': 64, 'num_blocks': 2, 'num_transforms': 4, 'num_bins': 6},
    device='cpu',
)
# Arguments: summaries and parameters of the training set, then of the validation set
val_loss, epochs = nle.fit(s_train, theta_train_nle, s_val, theta_val_nle, seed=0,
                           train={'lr': 1e-3, 'batch_size': 200, 'max_epochs': 250, 'stop_after_epochs': 20})
print(f'NLE: best validation loss {val_loss:.3f} after {epochs} epochs ({time.time() - start:.0f} s)')

plt.plot(nle.history)
plt.xlabel('Epoch')
plt.ylabel('Validation loss')
plt.title('NLE training')
plt.show()

# The learned log-likelihood of the observation's summaries for some parameters (returns a torch tensor)
print('log q(s_obs | theta = 0):', nle.log_likelihood(s_obs, np.zeros((1, 3))).item())

## 4. The Posterior of the Observation

**Prior.** `sk.Prior(low, high)` is a flat prior in a box. Additional density terms (e.g. `sk.Gaussian`) and constraints can be added to it.

**Sampling.** `sk.sample_posteriors` runs an ensemble MCMC (with the [`torchemcee`](https://github.com/D-Gebauer/torchemcee) package) for one or many observations. It is configured with `sk.MCMCConfig`. The most important settings:
- `nwalkers`: number of walkers per observation
- `init`: `'prior'` starts the walkers at random positions in the prior, `'ball'` in a small ball around `init_centre`
- `burn_in`, `block`, `max_steps`: after the burn-in, the chain grows in blocks of `block` steps until it has converged or `max_steps` is reached
- `rhat`, `min_ess`: the convergence criteria (see below)
- `n_samples`: number of posterior samples returned per observation

Unlike our sampler in Notebook 7, we do not have to guess the number of steps: the sampler **checks convergence** and stops by itself.

In [ ]:
prior = sk.Prior(LOGNORMAL_PRIOR_LOW, LOGNORMAL_PRIOR_HIGH, names=param_names)        # flat prior in [-2, 2]^3

config_obs = sk.MCMCConfig(nwalkers=64, init='prior', burn_in=500, block=500, max_steps=6000, n_samples=5000, seed=0)
posterior_obs = sk.sample_posteriors(nle, s_obs, prior, config_obs)

samples_obs = posterior_obs.samples[0]        # samples has the shape [num_observations, n_samples, num_parameters]
print('\nShape of the samples:', posterior_obs.samples.shape)
print('Converged:', posterior_obs.converged)

The line printed by the sampler and the dictionary `posterior.diagnostics` contain the convergence diagnostics for every observation:
- `rhat_max`: the Gelman-Rubin statistic $\hat{R}$ compares the scatter between groups of walkers with the scatter within them. It approaches one when all walkers sample the same distribution.
- `ess_min`: the effective sample size, i.e. the number of independent samples the chain is worth.
- `tau_max`: the autocorrelation time in steps.

`posterior.converged` tells us whether the thresholds of the configuration were reached.

As in Notebook 7, we compare with the exact posterior of the full data vector, which we can compute for this toy model.

In [ ]:
# Exact posterior, sampled with emcee (as in Notebook 7)
x_obs_np = x_obs.numpy().astype(np.float64)
theta_0 = np.random.uniform(-2, 2, (200000, 3))
theta_0 = theta_0[np.isfinite(exact_log_posterior(theta_0, x_obs_np))][:200]
sampler_exact = emcee.EnsembleSampler(200, 3, exact_log_posterior, args=(x_obs_np,), vectorize=True)
sampler_exact.run_mcmc(theta_0, 5000)
samples_exact = sampler_exact.get_chain(discard=1000, thin=10, flat=True)

print('Standard deviations of the posteriors (theta_1, theta_2, theta_3):')
print('Exact, full data          ', samples_exact.std(axis=0).round(3))
print('sbikit, compressed data   ', samples_obs.std(axis=0).round(3))

c = ChainConsumer()
labels = [rf"$\theta_{i+1}$" for i in range(3)]
c.add_chain(Chain(samples=pd.DataFrame(samples_obs, columns=labels), color='red', name="sbikit (VMIM + NLE)", sigmas=[0, 1, 2]))
c.add_chain(Chain(samples=pd.DataFrame(samples_exact, columns=labels), color='black', name="Exact (full data)", sigmas=[0, 1, 2]))
c.add_truth(Truth(location=dict(zip(labels, theta_obs.tolist()))))
c.set_plot_config(PlotConfig(contour_label_font_size=36, label_font_size=18, tick_font_size=16))
fig = c.plotter.plot()

The posterior shows the two peaks in $\theta_2$ and is somewhat wider than the exact posterior, as in Notebook 7: the compression loses some information.

## 5. Coverage Test

For the coverage test we need the posteriors of many test simulations. This is what `sample_posteriors` is built for: we pass all 300 compressed test simulations at once, and every one of them gets its own ensemble of walkers. One MCMC step evaluates the flow once for all of them (this is the batched sampler which we wrote by hand in Notebook 7).

The default thresholds ($\hat{R} < 1.01$ and 2000 effective samples for every observation, with 256 walkers) are designed for final results on a GPU. For a coverage test on a laptop we relax them.

In [ ]:
config_test = sk.MCMCConfig(nwalkers=32, init='prior', burn_in=300, block=300, max_steps=1200,
                            rhat=1.05, min_ess=400, n_samples=1000, seed=0)
start = time.time()
posterior_test = sk.sample_posteriors(nle, s_test, prior, config_test)
print(f'\nSampling time: {time.time() - start:.0f} s')
print('Shape of the samples:', posterior_test.samples.shape)
print(f'Converged: {posterior_test.converged.sum()} of {num_test} posteriors')

Not all chains have reached the thresholds within `max_steps`. The ones which have not are mostly posteriors with two well separated peaks in $\theta_2$, between which the walkers move only rarely. For a final result we would run longer. Here we continue and keep in mind that the convergence is not perfect.

### Metrics

`evaluate` computes a standard set of metrics from the posterior samples and the true parameters and returns them in two parts:
- `record`: a dictionary of numbers which can be saved as JSON,
- `arrays`: the TARP curves and values for every test simulation.

With `tarp_sets` we can run the TARP test for subsets of the parameters in addition to all of them together.

In [ ]:
record, arrays = evaluate(posterior_test.samples, theta_test, param_names,
                          tarp_sets={'theta2': ['theta2'], 'theta1_theta3': ['theta1', 'theta3']},
                          fom_sets={'theta1_theta3': ['theta1', 'theta3']})

plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Ideal')
plt.plot(arrays['tarp_alpha'], arrays['tarp_ecp_all'], label='All parameters')
plt.plot(arrays['tarp_alpha'], arrays['tarp_ecp_theta2'], label=r'$\theta_2$ only')
plt.plot(arrays['tarp_alpha'], arrays['tarp_ecp_theta1_theta3'], label=r'$\theta_1$ and $\theta_3$')
plt.xlabel('Credibility Level')
plt.ylabel('Expected Coverage')
plt.title('TARP Coverage Test (sbikit)')
plt.legend()
plt.show()

print('TARP                 bias               confidence        largest deviation')
for name, tarp in record['tarp'].items():
    print(f"{name:15s} {tarp['bias']:+.3f} +- {tarp['bias_sd']:.3f}    {tarp['confidence']:+.3f} +- {tarp['confidence_sd']:.3f}    {tarp['max_dev']:.3f}")

print('\nFraction of true values inside the central 68% / 95% interval:')
for name, marginal in record['marginal'].items():
    print(f"{name}: {marginal['cov68']:.2f} / {marginal['cov95']:.2f}")

The curves are close to the diagonal, as in Notebook 7. In addition to the plot, `sbikit` summarizes each curve in two numbers with bootstrap errors:
- **bias:** negative if the curve lies below the diagonal,
- **confidence:** negative if the posteriors are overconfident (too narrow), positive if they are too wide.

These numbers average over many random reference points and are therefore more sensitive than a look at a single curve. They show something the plot barely does: for $\theta_1$ and $\theta_3$ the confidence is consistent with zero, but for $\theta_2$ it is significantly negative. The posteriors of $\theta_2$ are mildly **overconfident**.

### Where Does the Overconfidence Come From?

The data only depend on $\theta_2^2$, so the true likelihood is exactly the same for $+\theta_2$ and $-\theta_2$, and the true posterior has two peaks of exactly equal weight. The flow does not know about this symmetry. It has to learn it from the simulations, and it does so only approximately. A small difference in the log-likelihood of the two peaks is enough to shift the weight noticeably from one to the other: the posterior then claims to know the sign of $\theta_2$ better than the data allow.

We can see this directly: for test simulations with a large $|\theta_2|$, we compute the fraction of posterior samples which have the same sign as the true $\theta_2$. It should be 0.5 for every single one of them.

In [ ]:
two_peaks = np.abs(theta_test[:, 1]) > 0.7
same_sign = np.sign(posterior_test.samples[two_peaks, :, 1]) == np.sign(theta_test[two_peaks, 1])[:, None]
fraction_same_sign = same_sign.mean(axis=1)

plt.figure(figsize=(7, 4))
plt.hist(fraction_same_sign, bins=20, range=(0, 1))
plt.axvline(0.5, color='black', ls='--', label='Exact symmetry')
plt.xlabel(r'Fraction of posterior samples with the correct sign of $\theta_2$')
plt.ylabel('Number of test simulations')
plt.legend()
plt.show()

The weights of the two peaks scatter widely around 0.5.

If we know about a symmetry of the model, we can enforce it. Here it is enough to mirror the samples in $\theta_2$ (in a real analysis one would rather build the symmetry into the model, e.g. by using $\theta_2^2$ as parameter):

In [ ]:
samples_mirrored = posterior_test.samples * np.array([1, -1, 1])
samples_symmetric = np.concatenate([posterior_test.samples, samples_mirrored], axis=1)

record_symmetric, _ = evaluate(samples_symmetric, theta_test, param_names, tarp_sets={'theta2': ['theta2']})
for name in ('all', 'theta2'):
    print(f"{name:7s} confidence before: {record['tarp'][name]['confidence']:+.3f} +- {record['tarp'][name]['confidence_sd']:.3f}"
          f"   after symmetrizing: {record_symmetric['tarp'][name]['confidence']:+.3f} +- {record_symmetric['tarp'][name]['confidence_sd']:.3f}")

Most of the overconfidence is gone. The lesson is a general one: a neural density estimator only learns the symmetries of a problem approximately, and multimodal posteriors are where this shows first. A coverage plot which looks fine by eye is not the end of the validation. Quantitative metrics with error bars, evaluated for subsets of the parameters, are what tells you where to look.

## 6. More Features

### Changing the Prior after Sampling

Since NLE learns the likelihood, the prior can be changed without retraining. If the new prior is not too different, we do not even have to run the MCMC again: `importance_weights` reweights existing samples. As an example, we add a Gaussian prior on $\theta_3$ (mean 0, standard deviation 0.1) to the flat prior.

In [ ]:
prior_new = sk.Prior(LOGNORMAL_PRIOR_LOW, LOGNORMAL_PRIOR_HIGH, names=param_names,
                     terms=[sk.Gaussian(dims=[2], mean=[0.0], cov=[[0.1**2]])])

weights = importance_weights(posterior_obs.samples, prior, prior_new)          # [num_observations, n_samples]
summary_flat = posterior_summary(posterior_obs.samples)
summary_new = posterior_summary(posterior_obs.samples, weights=weights)

print('Posterior of the observation   mean (theta_1, theta_2, theta_3)     std')
print('Flat prior                    ', summary_flat['mean'][0].round(3), '   ', summary_flat['std'][0].round(3))
print('With Gaussian prior on theta_3', summary_new['mean'][0].round(3), '   ', summary_new['std'][0].round(3))

### Saving and Loading

`sk.save` and `sk.load` store and restore compressors, likelihoods and priors. `sk.run_record` writes a JSON file with everything needed to reproduce a run: your configuration, the versions of the packages, the git commit, and the metrics.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    sk.save(compressor, f'{directory}/compressor.pt')
    sk.save(nle, f'{directory}/nle.pt')
    compressor_loaded = sk.load(f'{directory}/compressor.pt')
    nle_loaded = sk.load(f'{directory}/nle.pt')

    run = sk.run_record(f'{directory}/run.json', config={'flow': nle.flow_config, 'mcmc': 'see notebook'},
                        extra={'metrics': record})

print('Loaded compressor gives the same summaries:', np.allclose(compressor_loaded(x_obs[None]), s_obs))
print('Loaded likelihood gives the same values:   ',
      torch.allclose(nle_loaded.log_likelihood(s_obs, np.zeros((1, 3))), nle.log_likelihood(s_obs, np.zeros((1, 3)))))
print('Entries of the run record:', list(run.keys()))

## Cheat Sheet

| Step | Code |
|---|---|
| Compression | `compressor = VMIMCompressor(n_out=6, config={...})`<br>`compressor.fit(x_train, theta_train, x_val, theta_val)`<br>`s = compressor(x)` |
| Likelihood | `nle = sk.NLE(n_x=6, n_theta=3, flow={'model': 'nsf', ...})`<br>`nle.fit(s_train, theta_train, s_val, theta_val, train={...})` |
| Prior | `prior = sk.Prior(low, high, names=param_names)` |
| Sampling | `posterior = sk.sample_posteriors(nle, s_obs, prior, sk.MCMCConfig(nwalkers=64, init='prior'))`<br>`posterior.samples` `[num_observations, n_samples, P]`, `posterior.converged`, `posterior.diagnostics` |
| Metrics | `record, arrays = evaluate(posterior.samples, theta_true, param_names, tarp_sets={...})` |
| New prior | `weights = importance_weights(posterior.samples, prior, prior_new)` |
| Save | `sk.save(nle, 'nle.pt')`, `nle = sk.load('nle.pt')`, `sk.run_record('run.json', config=..., extra=...)` |

Things to keep in mind:
- Data come first, parameters second: `fit(x, theta, ...)`. The exceptions are `sk.Dataset(theta, x, ...)` and `evaluate(samples, truth, ...)`.
- Unknown keys in the `flow` and `train` dictionaries are ignored without a message, so check the spelling.
- The default MCMC settings are meant for GPUs. On a laptop, reduce `nwalkers` and relax `rhat` and `min_ess`, and always look at `posterior.converged`.

Not covered here (see the README): likelihood ensembles (`sk.Ensemble`), parameter transforms, hyperparameter searches with Optuna (`sbikit.hpt`), blinding, and the other compressors.

## Summary

- `sbikit` provides the pipeline of Notebook 7 (compression, NLE, batched ensemble MCMC, coverage) as a package, with NumPy arrays in and out.
- The sampler checks its own convergence. Posteriors with several peaks are the hardest to converge.
- `evaluate` turns the coverage test into numbers with error bars, for all parameters and for subsets. This revealed a mild overconfidence in $\theta_2$, caused by the approximately learned symmetry of the likelihood.
- Because the likelihood is learned, the prior can be changed afterwards, even by simply reweighting the samples.

**Next:** Notebook 9 puts all four approaches side by side on the Gaussian toy problem.